# 04 — Classical Machine Learning

**Goal**: Breadth of classical ML approaches. RF, SVM, boosting, discriminant analysis.  
**Note**: SVM on 630K is intractable; we use 50K representative subsample for kernel SVMs.


In [1]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow, json, pathlib, time

from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier,
    BaggingClassifier, HistGradientBoostingClassifier
)
from sklearn.svm import SVC, LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

from src.data_utils import load_data, get_X_y, FEATURE_COLS, TARGET
from src.evaluation import cv_evaluate, log_mlflow_run
from src.visualization import save_fig, PALETTE

sns.set_theme(style='whitegrid', palette=PALETTE)
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('Heart-Disease-Kaggle')

train = load_data('train')
X, y = get_X_y(train, extra_features=False)
ss = StandardScaler()
X_scaled = pd.DataFrame(ss.fit_transform(X), columns=X.columns, index=X.index)

# Sub-sample for slow models (SVM)
np.random.seed(42)
svm_idx = np.random.choice(len(X_scaled), size=50000, replace=False)
X_svm = X_scaled.iloc[svm_idx].reset_index(drop=True)
y_svm = y.iloc[svm_idx].reset_index(drop=True)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
CV_svm = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
RESULTS_DIR = pathlib.Path('../results/metrics')
all_results = []

def run_model(name, model, Xf, yf, cv, phase='classical_ml', feature_set='baseline', params=None, note=''):
    t0 = time.time()
    metrics = cv_evaluate(model, Xf, yf, cv=cv)
    elapsed = time.time() - t0
    log_mlflow_run(name, metrics, params=params or {}, tags={'phase': phase, 'feature_set': feature_set})
    result = {'model': name, 'phase': phase, **metrics, 'elapsed_s': round(elapsed,1), 'note': note}
    print(f'  {name:<45} AUC={metrics["roc_auc_mean"]:.4f}±{metrics["roc_auc_std"]:.4f}  '
          f'F1={metrics["f1_mean"]:.4f}  Recall={metrics["recall_mean"]:.4f}  [{elapsed:.0f}s]')
    return result

print(f'X full: {X.shape}, X SVM sample: {X_svm.shape}')

X full: (630000, 13), X SVM sample: (50000, 13)


## 4.1 Random Forest & Extra Trees

In [2]:
print('--- Random Forest & Extra Trees ---')
ensemble_models = [
    ('Random Forest (n=100)',    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
     {'n_estimators': 100}),
    ('Random Forest (n=300)',    RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
     {'n_estimators': 300}),
    ('Extra Trees (n=100)',      ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
     {'n_estimators': 100}),
    ('Extra Trees (n=300)',      ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=-1),
     {'n_estimators': 300}),
]
for name, model, params in ensemble_models:
    r = run_model(name, model, X, y, CV, params=params)
    all_results.append(r)

--- Random Forest & Extra Trees ---


  Random Forest (n=100)                         AUC=0.9470±0.0004  F1=0.8659  Recall=0.8590  [49s]


  Random Forest (n=300)                         AUC=0.9480±0.0004  F1=0.8668  Recall=0.8609  [179s]


  Extra Trees (n=100)                           AUC=0.9447±0.0005  F1=0.8620  Recall=0.8560  [55s]


  Extra Trees (n=300)                           AUC=0.9457±0.0005  F1=0.8632  Recall=0.8582  [165s]


## 4.2 Gradient Boosting (sklearn)

In [3]:
print('--- sklearn Gradient Boosting ---')
gb_models = [
    ('HistGradientBoosting (default)', HistGradientBoostingClassifier(random_state=42),
     {'type': 'HistGB', 'max_iter': 100}),
    ('HistGradientBoosting (lr=0.05)', HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=42),
     {'type': 'HistGB', 'lr': 0.05, 'max_iter': 300}),
    ('AdaBoost (n=100)',  AdaBoostClassifier(n_estimators=100, random_state=42),
     {'n_estimators': 100}),
    ('AdaBoost (n=200)',  AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=42),
     {'n_estimators': 200, 'lr': 0.5}),
    ('Bagging (DT base)', BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=5),
                                            n_estimators=100, random_state=42, n_jobs=-1),
     {'base': 'DT depth=5', 'n_estimators': 100}),
]
for name, model, params in gb_models:
    r = run_model(name, model, X, y, CV, params=params)
    all_results.append(r)

--- sklearn Gradient Boosting ---


  HistGradientBoosting (default)                AUC=0.9547±0.0004  F1=0.8739  Recall=0.8673  [11s]


  HistGradientBoosting (lr=0.05)                AUC=0.9550±0.0004  F1=0.8743  Recall=0.8675  [25s]


/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/ense

  AdaBoost (n=100)                              AUC=0.9543±0.0004  F1=0.8728  Recall=0.8611  [20s]


/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/Users/dustinober/Projects/Kaggle-Predicting-Heart-Disease/.venv/lib/python3.11/site-packages/sklearn/ense

  AdaBoost (n=200)                              AUC=0.9544±0.0004  F1=0.8728  Recall=0.8606  [49s]


  Bagging (DT base)                             AUC=0.9362±0.0009  F1=0.8463  Recall=0.8345  [37s]


## 4.3 Support Vector Machines (50K sample)

In [4]:
print('--- SVM (50K subsample) ---')
svm_models = [
    ('SVM (Linear)',  LinearSVC(C=1.0, max_iter=2000, random_state=42),
     {'kernel': 'linear', 'C': 1.0}, ['roc_auc', 'f1', 'recall', 'accuracy']),
    ('SVM (RBF, C=1)',   SVC(kernel='rbf',  C=1.0, gamma='scale', probability=True, random_state=42),
     {'kernel': 'rbf', 'C': 1.0, 'gamma': 'scale'}, None),
    ('SVM (RBF, C=10)',  SVC(kernel='rbf',  C=10,  gamma='scale', probability=True, random_state=42),
     {'kernel': 'rbf', 'C': 10, 'gamma': 'scale'}, None),
    ('SVM (Poly, d=3)',  SVC(kernel='poly', degree=3, probability=True, random_state=42),
     {'kernel': 'poly', 'degree': 3}, None),
]

from sklearn.model_selection import cross_validate as cv_fn
DEFAULT_SCORING = ['roc_auc', 'f1', 'recall', 'precision', 'accuracy']

for name, model, params, scoring in svm_models:
    use_scoring = scoring or DEFAULT_SCORING
    t0 = time.time()
    # LinearSVC doesn't support probability, so no roc_auc — handle gracefully
    try:
        metrics = cv_evaluate(model, X_svm, y_svm, cv=CV_svm, scoring=use_scoring)
    except Exception as e:
        use_scoring = ['f1', 'recall', 'precision', 'accuracy']
        metrics = cv_evaluate(model, X_svm, y_svm, cv=CV_svm, scoring=use_scoring)
        metrics['roc_auc_mean'] = 0.0
        metrics['roc_auc_std'] = 0.0
    elapsed = time.time() - t0
    log_mlflow_run(name, metrics, params=params, tags={'phase': 'classical_ml', 'note': 'sample=50K'})
    r = {'model': name, 'phase': 'classical_ml', **metrics, 'elapsed_s': round(elapsed,1), 'note': 'sample=50K'}
    all_results.append(r)
    print(f'  {name:<45} AUC={metrics["roc_auc_mean"]:.4f}  F1={metrics["f1_mean"]:.4f}  Recall={metrics["recall_mean"]:.4f}  [{elapsed:.0f}s]')

--- SVM (50K subsample) ---


  SVM (Linear)                                  AUC=0.9500  F1=0.8646  Recall=0.8520  [0s]


  SVM (RBF, C=1)                                AUC=0.9373  F1=0.8648  Recall=0.8555  [136s]


  SVM (RBF, C=10)                               AUC=0.9285  F1=0.8607  Recall=0.8511  [254s]


  SVM (Poly, d=3)                               AUC=0.9352  F1=0.8634  Recall=0.8495  [108s]


## 4.4 Discriminant Analysis

In [5]:
print('--- Discriminant Analysis ---')
da_models = [
    ('LDA',  LinearDiscriminantAnalysis(),              {'solver': 'svd'}),
    ('QDA',  QuadraticDiscriminantAnalysis(),           {'reg_param': 0.0}),
    ('QDA (reg=0.1)', QuadraticDiscriminantAnalysis(reg_param=0.1), {'reg_param': 0.1}),
]
for name, model, params in da_models:
    r = run_model(name, model, X_scaled, y, CV, params=params, feature_set='scaled')
    all_results.append(r)

--- Discriminant Analysis ---


  LDA                                           AUC=0.9490±0.0003  F1=0.8612  Recall=0.8389  [1s]


  QDA                                           AUC=0.9352±0.0004  F1=0.8541  Recall=0.8614  [1s]


  QDA (reg=0.1)                                 AUC=0.9382±0.0004  F1=0.8564  Recall=0.8556  [1s]


## 4.5 Results Summary

In [6]:
results_df = pd.DataFrame(all_results)
summary = results_df[['model', 'roc_auc_mean', 'roc_auc_std', 'f1_mean', 'recall_mean', 'precision_mean']].sort_values('roc_auc_mean', ascending=False)
print('\n=== CLASSICAL ML LEADERBOARD ===')
print(summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


=== CLASSICAL ML LEADERBOARD ===
                         model  roc_auc_mean  roc_auc_std  f1_mean  recall_mean  precision_mean
HistGradientBoosting (lr=0.05)        0.9550       0.0004   0.8743       0.8675          0.8813
HistGradientBoosting (default)        0.9547       0.0004   0.8739       0.8673          0.8806
              AdaBoost (n=200)        0.9544       0.0004   0.8728       0.8606          0.8853
              AdaBoost (n=100)        0.9543       0.0004   0.8728       0.8611          0.8849
                  SVM (Linear)        0.9500       0.0035   0.8646       0.8520             NaN
                           LDA        0.9490       0.0003   0.8612       0.8389          0.8846
         Random Forest (n=300)        0.9480       0.0004   0.8668       0.8609          0.8729
         Random Forest (n=100)        0.9470       0.0004   0.8659       0.8590          0.8728
           Extra Trees (n=300)        0.9457       0.0005   0.8632       0.8582          0.8682
      

In [7]:
# Visualization
plot_df = results_df.sort_values('roc_auc_mean', ascending=True)
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
metrics_plot = [('roc_auc_mean', 'roc_auc_std', 'ROC-AUC'), ('f1_mean', 'f1_std', 'F1-Score'), ('recall_mean', 'recall_std', 'Recall')]
for ax, (mc, sc, title) in zip(axes, metrics_plot):
    colors = sns.color_palette(PALETTE, n_colors=len(plot_df))
    bars = ax.barh(plot_df['model'], plot_df[mc], xerr=plot_df[sc], color=colors, capsize=3, error_kw={'elinewidth': 1})
    ax.set_xlabel(title); ax.set_title(f'{title} (5-fold CV)')
    ax.set_xlim(max(0, plot_df[mc].min() - 0.05), 1.0)
    for bar, val in zip(bars, plot_df[mc]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=7)
fig.suptitle('Classical ML Models — Cross-Validation Performance', fontsize=13)
fig.tight_layout()
save_fig('04_classical_ml_results', fig)
plt.show()

In [8]:
results_df.to_csv(RESULTS_DIR / '04_classical_ml_results.csv', index=False)
print(f'Saved. Best classical model: {summary.iloc[0]["model"]}  AUC={summary.iloc[0]["roc_auc_mean"]:.4f}')

Saved. Best classical model: HistGradientBoosting (lr=0.05)  AUC=0.9550
